# 07.01 · Thinking in LangGraph

**Module 07 · LangGraph Fundamentals** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 60–90 min |
| 📋 Prerequisites | Module 02 (`create_agent`), Lesson 01.04 (structured output). |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Decompose** a process into discrete steps, and label each one: LLM step, data step, action step, user-input step.
2. **Design the state** before writing nodes: what each step reads and writes, kept raw (prompts are formatted on demand).
3. **Sketch** the graph as a Mermaid diagram in a Markdown cell, then build it and compare.
4. **Decide** when `create_agent` is enough and when to drop down to LangGraph.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph) | The five design steps this lesson follows (an email-support agent in the docs; a ticket triage here) |
| [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview) | What LangGraph adds: durable execution, human-in-the-loop, memory, streaming, debugging |
| [LangGraph quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart) | The smallest Graph API and Functional API examples |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langchain", "langchain-core",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/07-langgraph-fundamentals/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
LANGSMITH_PROJECT = 'm07-thinking-in-langgraph'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # traces (if LANGSMITH_TRACING is on) go here, unless .env sets one

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (07.07 reuses Module 06): no usage telemetry.
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

---
## Part 1 · Start with the process, not the code

The docs' advice: write down the process you want to automate as if you were explaining it to a new colleague. Ours: **support-ticket triage** for a small SaaS (the 12 tickets of Lesson 01.03).

> A ticket arrives. Decide its category (billing, bug, feature request, account, other) and whether it is urgent. Urgent tickets go to a human right away. The rest get a drafted reply in the right tone; bug reports also get a short bug summary for the engineers. Every ticket ends with a record of what happened.

**Step 1 · Map the steps.** Each box is one thing that happens:

| Step | Kind | Reads | Writes |
|---|---|---|---|
| `classify` | **LLM step** (structured output) | ticket text | category, urgent |
| `escalate` | **action step** (would page a human; here: write to a log) | ticket, category | outcome |
| `summarize_bug` | **LLM step** | ticket text | bug summary |
| `draft_reply` | **LLM step** | ticket, category, bug summary | reply |
| `record` | **data step** | everything | outcome |

A fifth kind, **user-input steps** (wait for a human to approve), arrives with interrupts in Module 08.

**Step 2 · What each step needs.** `classify` needs only the text. `draft_reply` needs the category (tone) and, for bugs, the summary. Nothing needs the whole conversation history, so we don't need `MessagesState` here.

**Step 3 · Sketch the graph (Mermaid, before any code).** This is a Markdown cell: JupyterLab 4.1+ and GitHub render ```` ```mermaid ```` blocks as diagrams. Double-click the cell to see the source.

```mermaid
flowchart TD
    START([START]) --> classify
    classify -->|urgent| escalate
    classify -->|bug| summarize_bug
    classify -->|other categories| draft_reply
    summarize_bug --> draft_reply
    escalate --> record
    draft_reply --> record
    record --> END([END])
```

No diagram, just text? Your JupyterLab is older than 4.1 (`jupyter lab --version`): the text is still a fine design document. VS Code renders Mermaid with the *Markdown Preview Mermaid Support* extension.

**Step 4 · Design the state.** Keep it **raw**: store the category and the ticket, not a formatted prompt. Prompts are built inside the node that needs them, so you can change a prompt without migrating saved state.

In [ ]:
# 🧪 Offline: the state and the structured output for the LLM step.
from typing import Literal, TypedDict

from pydantic import BaseModel, Field

Category = Literal["billing", "bug", "feature_request", "account", "other"]


class Triage(BaseModel):
    """How to handle one support ticket."""
    category: Category
    urgent: bool = Field(description="True only if the customer is blocked, losing money, or data is at risk")


class TicketState(TypedDict, total=False):
    ticket_id: str
    text: str
    category: Category
    urgent: bool
    bug_summary: str
    reply: str
    outcome: str


print(list(TicketState.__annotations__))

**You should now see** `['ticket_id', 'text', 'category', 'urgent', 'bug_summary', 'reply', 'outcome']`. `total=False` means every key is optional: nodes fill them in as the ticket moves through the graph.

---
## Part 2 · Build the nodes

**Step 5 · Nodes and wiring.** A node is a plain function: it receives the **current state** and returns a **partial update** (only the keys it changed). Offline, a scripted model stands in for the LLM: it answers `Triage` with simple keyword rules, so you can see the graph work without a key.

In [ ]:
# 🧪 Offline: a scripted "classifier" (keyword rules) that answers like a model with structured output.
import json

from langchain_core.messages import AIMessage

from mastery.testing import ScriptedChatModel

RULES = {"billing": ("charged", "invoice", "refund", "price"), "bug": ("crash", "error", "does nothing", "broken"),
         "account": ("log in", "password", "account"), "feature_request": ("could you add", "would be nice", "please add")}
URGENT = ("crash", "charged twice", "can't log in", "data", "lost")


def keyword_triage(messages):
    text = messages[-1].text.lower()
    category = next((c for c, words in RULES.items() if any(w in text for w in words)), "other")
    triage = Triage(category=category, urgent=any(w in text for w in URGENT))
    return AIMessage("", tool_calls=[{"name": "Triage", "id": "t1", "args": triage.model_dump()}])


def scripted_writer(messages):
    return AIMessage("Thanks for writing in. " + messages[-1].text.split("Ticket:")[-1].strip()[:60] + "... We're on it.")


offline_classifier = ScriptedChatModel(responses=[keyword_triage], repeat_last=True)
offline_writer = ScriptedChatModel(responses=[scripted_writer], repeat_last=True)
TICKETS = [json.loads(line) for line in (REPO_ROOT / "data" / "samples" / "support-tickets.jsonl").read_text(encoding="utf-8").splitlines()]
print(len(TICKETS), "tickets |", TICKETS[1]["text"])

In [ ]:
# 🧪 Offline: the five nodes and the wiring from the Mermaid sketch.
from langgraph.graph import END, START, StateGraph

AUDIT: list[str] = []   # stands in for a ticketing system


def build_triage_graph(classifier, writer):
    triage_model = classifier.with_structured_output(Triage)

    def classify(state: TicketState) -> dict:
        result = triage_model.invoke(f"Classify this support ticket.\nTicket: {state['text']}")
        return {"category": result.category, "urgent": result.urgent}

    def escalate(state: TicketState) -> dict:
        AUDIT.append(f"PAGE on-call: {state['ticket_id']} ({state['category']})")
        return {"outcome": "escalated to a human"}

    def summarize_bug(state: TicketState) -> dict:
        reply = writer.invoke(f"Summarize this bug report in one line for engineers.\nTicket: {state['text']}")
        return {"bug_summary": reply.text}

    def draft_reply(state: TicketState) -> dict:
        prompt = f"Write a short, kind reply for a {state['category']} ticket.\nTicket: {state['text']}"
        return {"reply": writer.invoke(prompt).text}

    def record(state: TicketState) -> dict:
        outcome = state.get("outcome") or f"replied ({state['category']})"
        AUDIT.append(f"{state['ticket_id']}: {outcome}")
        return {"outcome": outcome}

    def route(state: TicketState) -> str:
        if state["urgent"]:
            return "escalate"
        return "summarize_bug" if state["category"] == "bug" else "draft_reply"

    builder = StateGraph(TicketState)
    for node in (classify, escalate, summarize_bug, draft_reply, record):
        builder.add_node(node)                       # the node name is the function name
    builder.add_edge(START, "classify")
    builder.add_conditional_edges("classify", route, ["escalate", "summarize_bug", "draft_reply"])
    builder.add_edge("summarize_bug", "draft_reply")
    builder.add_edge("escalate", "record")
    builder.add_edge("draft_reply", "record")
    builder.add_edge("record", END)
    return builder.compile()


triage_graph = build_triage_graph(offline_classifier, offline_writer)
for ticket in TICKETS[:6]:
    out = triage_graph.invoke({"ticket_id": ticket["id"], "text": ticket["text"]})
    print(f"{ticket['id']} {out['category']:<16} urgent={out['urgent']!s:<5} → {out['outcome']}")

**You should now see** six rows: `T01 billing urgent=True → escalated to a human` ("charged twice"), `T02 bug urgent=False → replied (bug)`, … `T06 bug urgent=True → escalated` (a crash). The keyword rules are crude on purpose: the **graph** is what we're studying, and it doesn't change when a real model replaces the rules.

### What just happened?

`invoke` started at `START`, ran `classify`, asked `route` which edge to take, and kept going until `END`. After each node, LangGraph merged the node's partial update into the state (here every key simply overwrites; reducers in 07.02 change that). The **edges** encode the process you wrote in plain English; the **nodes** don't know about each other.

In [ ]:
# 🧪 Offline: compare the drawn graph with your Mermaid sketch (text form, no network).
print(triage_graph.get_graph().draw_mermaid())

**You should now see** Mermaid text with the same five nodes, the three dotted conditional edges out of `classify`, and `__start__`/`__end__`. 07.02 shows how to render it as an image.

---
## Part 3 · The same triage with a real model

> 💸 Calls `gpt-oss-20b` a few times (one per model node), a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 Live: swap the scripted models for gpt-oss-20b. The graph code is unchanged.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    live_graph = build_triage_graph(llm, llm)
    for ticket in TICKETS[:4]:
        out = live_graph.invoke({"ticket_id": ticket["id"], "text": ticket["text"]})
        print(f"{ticket['id']} {out['category']:<16} urgent={out['urgent']!s:<5} (label: {ticket['label']}) → {out['outcome']}")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) four rows whose categories match the labels. "Urgent" is a judgement call: compare with the keyword version and decide which you'd trust.

---
## Part 4 · `create_agent` or LangGraph?

`create_agent` (Module 02) **is** a LangGraph graph: a model node and a tools node in a loop. You drop down to your own graph when you need control the loop doesn't give you:

| You need… | `create_agent` | Your own graph |
|---|---|---|
| The model decides which tools to call, in any order | ✅ the point of it | possible, more code |
| A **fixed order** of steps (classify → reply → record) | prompt and hope | ✅ edges guarantee it |
| Deterministic routing on data (`urgent` → escalate) | middleware (Module 04) | ✅ a conditional edge |
| Different models / prompts per step | one model (+ middleware) | ✅ per node |
| Parallel branches, map-reduce | only parallel **tool calls** (each runs as its own task) | ✅ any steps, with `Send` (07.03) |
| Loops with a hard stop you can read | `ToolCallLimit` / `ModelCallLimit` | ✅ a counter in state (07.04) |

Rule of thumb: **if you can draw the process before it runs, draw it as a graph.** If only the model can decide the steps, use an agent, possibly as one node of a bigger graph (07.04, 07.07).

## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `InvalidUpdateError: Expected dict, got ...` | A node returned a string or a full object instead of a dict update | Return `{"key": value}` with only the keys you changed |
| `KeyError: 'category'` in a node | A node reads a key no earlier node wrote (`total=False` makes keys optional) | Use `state.get("key")`, or fix the edge order |
| Mermaid cell shows plain text | JupyterLab < 4.1 or classic Notebook | `jupyter lab --version`; upgrade in **base** (`conda update jupyterlab`) or read it as text |
| ``ValueError: Found edge ending at unknown node `x` `` | Typo in an edge name | Node names are the function names unless you pass one: `add_node("name", fn)` |

## ⚠️ Production notes

> ⚠️ **Production note: action steps must be idempotent.**
> `escalate` pages a human. If the run is retried (Module 08 adds durable execution), it may run twice. Use an idempotency key (the ticket id) in the real paging/ticketing call.

> ⚠️ **Production note: keep state raw and small.**
> State is saved at every step once you add a checkpointer. Store ids, categories and short texts; keep big documents in a store or database and put the key in state.

## 🏋️ Exercises

1. **(Easy)** Run the offline graph on all 12 tickets and count how many categories match `ticket["label"]`.
2. **(Medium)** Add a `feature_request` path: a node `log_feature` that appends to `AUDIT` and **skips** the reply. Update the Mermaid sketch first, then the code.
3. **(Medium)** Which of the five steps would you test with a unit test, and which with an eval? Write one sentence each.
4. **(Stretch, 💸)** Use the live classifier on all 12 tickets and compute accuracy against the labels. Is it better than the keyword rules?

<details>
<summary><b>Show solutions</b></summary>

**1.** `sum(triage_graph.invoke({"ticket_id": t["id"], "text": t["text"]})["category"] == t["label"] for t in TICKETS)`: **8/12** with these rules. The four misses (T07, T10, T11, T12) use words the rules don't know ("would be great", "shows yesterday's numbers", "email address", "receipt"): exactly why a model does this step.

**2.** In `route`: `if state["category"] == "feature_request": return "log_feature"`; add the node (it must return `{"outcome": "logged (feature_request)"}` too, or `record` writes "replied ..."), `builder.add_edge("log_feature", "record")`, and add `"log_feature"` to the list in `add_conditional_edges`.

**3.** Unit tests: `route` and `record` (pure logic, deterministic). Evals: `classify` and `draft_reply` (model quality; Module 10).

**4.** Loop over `TICKETS` with `live_graph`; typically 11–12/12 for the category. Urgency disagreements are the interesting part.
</details>

## ✅ Checkpoint

- [ ] I can list a process's steps and say which are LLM, data, action or user-input steps.
- [ ] I designed the state before the nodes and kept it raw.
- [ ] I wrote a Mermaid sketch and compared it with `draw_mermaid()`.
- [ ] I can explain when `create_agent` is enough.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Docs drill (10 min)

When the model asks for three tool calls in one message, how does `create_agent` run them? Find the answer in `langchain/agents/factory.py`.

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. `import langchain.agents.factory as f, inspect; print(inspect.getsourcefile(f))`, search `Send(`.
2. The routing after the model returns one `Send("tools", [tool_call])` **per pending call**: the `tools` node runs once per call, in parallel, in the same superstep.
3. So three calls = three `tools` tasks in one step, and their `ToolMessage`s are merged by the `messages` reducer.

</details>

## 🔎 Research checkpoint

The docs' [Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph) page designs an email agent. Compare its state with ours: which keys did they keep raw, and what did they decide **not** to store? Log one design rule you'll reuse in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Design first: steps → what each needs → state → nodes → wiring. A Mermaid sketch is cheap and reviewable.
- Nodes are functions returning partial updates; edges encode the process; conditional edges route on state.
- `create_agent` is a graph too. Write your own graph when you can draw the process before it runs.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 07.02 · Graph API: state, nodes, edges, reducers** → [`07-02-graph-api-basics.ipynb`](07-02-graph-api-basics.ipynb).

[Module 07 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)